# 8-K Material Event Alerts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/sec-edgar-filings-monitor.ipynb)

**What:** New SEC 8-K filings as an alert feed: every item decoded into plain English and ranked high, medium or low, with a severity filter and new-only monitor mode. Bankruptcy, cyber, restatement, delisting alerts. Also 10-K, IPO, full-text.

**Why:** run the [`sec-edgar-filings-monitor`](https://apify.com/transparent_meteorite/sec-edgar-filings-monitor) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Public SEC filing metadata only; no personal data is displayed.

## 1. Setup

In [ ]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/sec-edgar-filings-monitor) for every option.

In [ ]:
run_input = {'formTypes': ['8-K'], 'filedWithinDays': 7, 'maxItems': 10}

## 3. Run the actor and read the dataset

In [ ]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/sec-edgar-filings-monitor').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

## 4. Tabulate

Public SEC filing metadata only; no personal data is displayed.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["filedAt", "formType", "companyName", "tickers", "exchange", "sicDescription", "eightKItemLabels", "isAmendment"] if c in df.columns]
df[SHOW]

## 5. A quick analysis

In [ ]:
print(df['companyName'].value_counts().head(10))
print(df['isAmendment'].value_counts(dropna=False))
df[['companyName','eightKItemLabels']].head(10)

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [sec-edgar-filings-monitor on the Apify Store](https://apify.com/transparent_meteorite/sec-edgar-filings-monitor).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.